# HachimiMT — Chẩn đoán tốc độ (Colab vs Kaggle, cùng T4)

Colab chậm ~2.3× Kaggle dù cùng T4. Notebook này TÁCH thời gian: tokenize (CPU)
vs translate_batch (GPU) vs tổng app — để biết nút thắt là GPU (Colab T4 throttle)
hay CPU/overhead (Colab 2 vCPU yếu). Ép 1 GPU 2 nơi để so công bằng.

**Chạy trên CẢ Colab VÀ Kaggle** (GPU T4, Internet on), báo lại bảng mỗi nơi.

In [ ]:
# 1. Tải code + cài + ÉP 1 GPU (so công bằng Colab vs Kaggle)
import os
os.environ["HACHIMIMT_GPU_INDICES"] = "0"   # chỉ GPU 0 cả 2 nơi
import urllib.request, zipfile, shutil
ZIP_URL = "https://huggingface.co/spaces/ngocdang83/HachimiMT-demo/resolve/main/hachimimt-local.zip"
urllib.request.urlretrieve(ZIP_URL, "hachimimt-local.zip")
shutil.rmtree("hachimimt", ignore_errors=True)
with zipfile.ZipFile("hachimimt-local.zip") as z: z.extractall(".")
!pip install -q -r hachimimt/requirements.txt
import subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,clocks.max.sm,power.limit","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
print("vCPU:", os.cpu_count())

In [ ]:
# 2. Văn bản test + load model
import sys, time
sys.path.insert(0, os.path.abspath("hachimimt/src"))
from translator import HachimiTranslator, Backend, DEFAULT_MODEL_KEY
from hardware import detect_hardware_profile
from token_chunker import split_for_translation
PARA = (
    "他抬头看向远处的山门，心中升起一股莫名的悸动。师兄说得对，我等修士当以大道为重。"
    "她转身离去，没有再回头看一眼，仿佛一切已成定局。凌伊山掏出手机，查询起了机票。"
    "玄中门欲炼的丹药乃是凝婴丹，主材之一便是天婴果。他必须得抓紧时间了，否则来不及。"
)
TEXT = PARA * 500
tr = HachimiTranslator(detect_hardware_profile())
tr.load(DEFAULT_MODEL_KEY, Backend.CT2)
print("profile:", tr._profile.summary)
print("workers:", tr._ct2_worker_count, "· gpu:", tr._ct2_device_indices_label)

In [ ]:
# 3. TÁCH thời gian: tokenize (CPU) vs translate_batch (GPU) vs tổng app
import time
chunks = split_for_translation(tr._tokenizer, TEXT, max_tokens=256, chunk_mode="sentence")
n = len(chunks)

# warmup
tr.translate_text(TEXT[:2000], chunk_mode="sentence", beam_size=1)

# (a) tokenize thuần (CPU)
t0 = time.perf_counter()
tokenized = tr._source_tokens_batch(chunks)
t_tok = time.perf_counter() - t0

# (b) translate_batch thuần (GPU, token đã sẵn)
import ctranslate2
opts = dict(max_batch_size=96*256, batch_type="tokens", beam_size=1,
            no_repeat_ngram_size=2, repetition_penalty=1.2)
t0 = time.perf_counter()
tr._ct2_model.translate_batch(tokenized, **opts)
t_gpu = time.perf_counter() - t0

# (c) tổng app (translate_text: chunk+tokenize+window+decode)
t0 = time.perf_counter()
tr.translate_text(TEXT, chunk_mode="sentence", beam_size=1)
t_app = time.perf_counter() - t0

print(f"Văn bản: {n} chunk, {len(TEXT):,} ký tự\n")
print(f"(a) tokenize THUẦN (CPU)     : {t_tok:.2f}s  ({n/t_tok:,.0f} chunk/s)")
print(f"(b) translate_batch THUẦN(GPU): {t_gpu:.2f}s  ({n/t_gpu:,.0f} chunk/s)")
print(f"(c) TỔNG app (translate_text) : {t_app:.2f}s  ({n/t_app:,.0f} chunk/s)")
print(f"\n→ overhead app = {t_app - t_gpu:.2f}s ({(t_app-t_gpu)/t_app*100:.0f}% tổng)")
print("  Nếu GPU(b) Colab≈Kaggle nhưng tổng(c) chậm → nút thắt CPU/overhead (CPU yếu).")
print("  Nếu GPU(b) Colab >> Kaggle (chậm hơn) → Colab T4 bị throttle (ngoài tầm app).")